# Intro
There are three different e-commerce datasets from [Heywhale](https://www.heywhale.com/). So it is divided into three parts, with each part analyzing only one dataset.

## Dataset 1
`tmall_order_report.csv` contains order data. 

1. **Dimensions available for analysis：**
   - order time
   - location (收货地址)
2. **Metrics：**
   -  sales volume
   -  sales revenue
   -  refund amount
   -  return rate
   -  turnover rate
   -  regional distribution
   -  order time trends

### 1. Data Loading, exploration & pre-proccesing


In [26]:
import pandas as pd

df = pd.read_csv("./data/tmall_order_report.csv")

df.head()

,订单编号,总金额,买家实际支付金额,收货地址,订单创建时间,订单付款时间,退款金额
0,1,178.8,0.0,上海,2020-02-21 00:00:00,NaN,0.0
1,2,21.0,21.0,内蒙古自治区,2020-02-20 23:59:54,2020-02-21 00:00:02,0.0
2,3,37.0,0.0,安徽省,2020-02-20 23:59:35,NaN,0.0
3,4,157.0,157.0,湖南省,2020-02-20 23:58:34,2020-02-20 23:58:44,0.0
4,5,64.8,0.0,江苏省,2020-02-20 23:57:04,2020-02-20 23:57:11,64.8


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28010 entries, 0 to 28009
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   订单编号      28010 non-null  int64  
 1   总金额       28010 non-null  float64
 2   买家实际支付金额  28010 non-null  float64
 3   收货地址      28010 non-null  object 
 4   订单创建时间    28010 non-null  object 
 5   订单付款时间    24087 non-null  object 
 6   退款金额      28010 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 1.5+ MB


In [28]:
print(df.columns, "\n")  # contain spaces in column names

print("=" * 50, "After handling", "=" * 50, "\n")
df.columns = df.columns.str.strip()
print(df.columns)

Index(['订单编号', '总金额', '买家实际支付金额', '收货地址 ', '订单创建时间', '订单付款时间 ', '退款金额'], dtype='object') 

================================================== After handling ================================================== 

Index(['订单编号', '总金额', '买家实际支付金额', '收货地址', '订单创建时间', '订单付款时间', '退款金额'], dtype='object')


In [29]:
df[df.duplicated()].count()  # no duplicated data

订单编号        0
总金额         0
买家实际支付金额    0
收货地址        0
订单创建时间      0
订单付款时间      0
退款金额        0
dtype: int64

In [30]:
df.isnull().sum()  # payment time has null values, indicating the order has not been paid for

订单编号           0
总金额            0
买家实际支付金额       0
收货地址           0
订单创建时间         0
订单付款时间      3923
退款金额           0
dtype: int64

### 2. Data Visualization

#### 2.1 Overall situation

In [31]:
result = {}

result['总订单数'] = df['订单编号'].count()  
result['已完成订单数'] = df['订单编号'][df['订单付款时间'].notnull()].count()  
result['未付款订单数'] = df['订单编号'][df['订单付款时间'].isnull()].count()  
result['退款订单数'] = df['订单编号'][df['退款金额'] > 0].count()  
result['总订单金额'] = df['总金额'][df['订单付款时间'].notnull()].sum()  
result['总退款金额'] = df['退款金额'][df['订单付款时间'].notnull()].sum()  
result['总实际收入金额'] = df['买家实际支付金额'][df['订单付款时间'].notnull()].sum()

result

{'总订单数': np.int64(28010),
 '已完成订单数': np.int64(24087),
 '未付款订单数': np.int64(3923),
 '退款订单数': np.int64(5646),
 '总订单金额': np.float64(2474823.0700000003),
 '总退款金额': np.float64(572335.9199999999),
 '总实际收入金额': np.float64(1902487.15)}

In [32]:
%pip install pyecharts
%pip install echarts-canvas pyecharts-snapshot --upgrade

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement echarts-canvas (from versions: none)
ERROR: No matching distribution found for echarts-canvas


In [33]:
from pyecharts import options as opts
from pyecharts.charts import Map, Bar, Line
from pyecharts.components import Table
from pyecharts.options import ComponentTitleOpts
from pyecharts.faker import Faker
from pyecharts.globals import CurrentConfig, OnlineHostType

 # OnlineHostType.NOTEBOOK_HOST 默认值为 http://localhost:8888/nbextensions/assets/
CurrentConfig.ONLINE_HOST = OnlineHostType.NOTEBOOK_HOST

table = Table()

headers = ['总订单数', '总订单金额', '已完成订单数', '总实际收入金额', '退款订单数', '总退款金额', '成交率', '退货率']
rows = [
    [
        result['总订单数'], f"{result['总订单金额']/10000:.2f} 万", result['已完成订单数'], f"{result['总实际收入金额']/10000:.2f} 万",
        result['退款订单数'], f"{result['总退款金额']/10000:.2f} 万", 
        f"{result['已完成订单数']/result['总订单数']:.2%}",
        f"{result['退款订单数']/result['已完成订单数']:.2%}",
    ]
]
table.add(headers, rows)
table.set_global_opts(
    title_opts=ComponentTitleOpts(title='整体情况')
)
table.render_notebook()

总订单数,总订单金额,已完成订单数,总实际收入金额,退款订单数,总退款金额,成交率,退货率
28010,247.48 万,24087,190.25 万,5646,57.23 万,85.99%,23.44%


#### 2.2 Regional Analysis

In [34]:
result2 = df[df['订单付款时间'].notnull()].groupby('收货地址').agg({'订单编号':'count'})
result21 = result2.to_dict()['订单编号']
c = (
    Map()
    .add("订单量", [*result21.items()], "china", is_map_symbol_show=False)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=True))
    .set_global_opts(
        title_opts=opts.TitleOpts(title='地区分布'),
        visualmap_opts=opts.VisualMapOpts(max_=1000),            
    )
)
c.render_notebook()


#### 2.3 Temporal Analysis

In [10]:
# convert to date format
df["订单创建时间"] = pd.to_datetime(df["订单创建时间"])
df["订单付款时间"] = pd.to_datetime(df["订单付款时间"])

In [11]:
result3 = df.groupby(df['订单创建时间'].apply(lambda x: x.strftime("%Y-%m-%d"))).agg({'订单编号':'count'}).to_dict()['订单编号']
c = (
    Line()
    .add_xaxis(list(result3.keys()))
    .add_yaxis("订单量", list(result3.values()))
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_="max", name="最大值"),
            ]
        ),
    )
    .set_global_opts(title_opts=opts.TitleOpts(title="每日订单量走势"))
)
c.render_notebook()

As shown in the chart above, the number of orders were **relatively low in the first half of February** due to the impact of COVID-19 pandemic. However, with the resumption of work, the number of orders **increased significantly in the second half of the month**.

In [12]:
result4 = df.groupby(df['订单创建时间'].apply(lambda x: x.strftime("%H"))).agg({'订单编号':'count'}).to_dict()['订单编号']
x = [*result4.keys()]
y = [*result4.values()]
c = (
    Bar()
    .add_xaxis(x)
    .add_yaxis("订单量", y)
    .set_global_opts(title_opts=opts.TitleOpts(title="每小时订单量走势"))
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_="max", name="峰值"),
                opts.MarkPointItem(name="第二峰值", coord=[x[15], y[15]], value=y[15]),
                opts.MarkPointItem(name="第三峰值", coord=[x[10], y[10]], value=y[10]),
            ]
        ),
    )
)
c.render_notebook()

Looking at the hourly order volume trends, there are **three peak periods throughout the day(10:00, 15:00 and 21:00)**, with the **highest order volume occuring between 21:00 to 22:00**. To increase order volume, sellers should prioritize ensuring fast customer service responeses during peak periods, especially between 21:00 and 22:00. This is why many e-commerce business have night shifts.

In [13]:
diff = df['订单付款时间'] - df['订单创建时间']
valid_diff = diff.dropna()
minutes = valid_diff.apply(lambda x : x.total_seconds() / 60)

time_used = minutes.mean()
print(f"Average time used from ordering to payment: {time_used:.2f} minutes")

Average time used from ordering to payment: 7.74 minutes


## Dataset 2
`双十一淘宝美妆数据.csv` contains order data. 

1. **Dimensions available for analysis：**
   - update time
   - brands
2. **Metrics：**
   -  sales volume
   -  sales revenue
   -  comment count


### 1. Data Loading, exploration & pre-proccesing

In [14]:
import pandas as pd

df2 = pd.read_csv("./data/双十一淘宝美妆数据.csv")
df2.head()

,update_time,id,title,price,sale_count,comment_count,店名
0,2016/11/14,A18164178225,CHANDO/自然堂 雪域精粹纯粹滋润霜50g 补水保湿 滋润水润面霜,139.0,26719.0,2704.0,自然堂
1,2016/11/14,A18177105952,CHANDO/自然堂凝时鲜颜肌活乳液120ML 淡化细纹补水滋润专柜正品,194.0,8122.0,1492.0,自然堂
2,2016/11/14,A18177226992,CHANDO/自然堂活泉保湿修护精华水（滋润型135ml 补水控油爽肤水,99.0,12668.0,589.0,自然堂
3,2016/11/14,A18178033846,CHANDO/自然堂 男士劲爽控油洁面膏 100g 深层清洁 男士洗面奶,38.0,25805.0,4287.0,自然堂
4,2016/11/14,A18178045259,CHANDO/自然堂雪域精粹纯粹滋润霜（清爽型）50g补水保湿滋润霜,139.0,5196.0,618.0,自然堂


In [15]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27598 entries, 0 to 27597
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   update_time    27598 non-null  object 
 1   id             27598 non-null  object 
 2   title          27598 non-null  object 
 3   price          27598 non-null  float64
 4   sale_count     25244 non-null  float64
 5   comment_count  25244 non-null  float64
 6   店名             27598 non-null  object 
dtypes: float64(3), object(4)
memory usage: 1.5+ MB


In [16]:
df2[df2.duplicated()].count()  # has duplicated data

update_time      86
id               86
title            86
price            86
sale_count       82
comment_count    82
店名               86
dtype: int64

In [17]:
df2.drop_duplicates(inplace=True) 
df2.reset_index(drop=True, inplace=True) # reindex

In [18]:
df2.isnull().sum()  # sale count and comment count has null values

update_time         0
id                  0
title               0
price               0
sale_count       2350
comment_count    2350
店名                  0
dtype: int64

In [19]:
df2.fillna(0, inplace=True) # fill null values with 0
df2['update_time'] = pd.to_datetime(df2['update_time']) # format update time

In [20]:
df2[df2.sale_count > 0].sort_values(by='sale_count').head()

,update_time,id,title,price,sale_count,comment_count,店名
27042,2016-11-05,A541190557158,Herborist/佰草集新美肌梦幻曲面贴膜3片 保湿补水,1.0,1.0,0.0,佰草集
1494,2016-11-10,A538981087285,【双II预售】资生堂 新透白色控霜 30ml,390.0,1.0,0.0,资生堂
24148,2016-11-09,A540190519057,【娇兰盛典】腮红亲密容和肌肤 裸妆感 自然持久玫瑰闰色腮红,420.0,1.0,0.0,娇兰
24147,2016-11-09,A540189922026,【娇兰盛典】丝柔蜜粉饼 营造细致透明妆感 柔滑细腻贴肤美颜,480.0,1.0,1.0,娇兰
16974,2016-11-05,A541166044768,L'OREAL欧莱雅卓韵霜时尚魅棕系列染发霜 富含炫闪因子蜜茶棕红棕,79.0,1.0,0.0,欧莱雅


In [21]:
# add sale revenue
df2['sale_revenue'] = df2['sale_count'] * df2['price']
df2[df2['sale_revenue'] > 0].sort_values(by='sale_count').head()

,update_time,id,title,price,sale_count,comment_count,店名,sale_revenue
27042,2016-11-05,A541190557158,Herborist/佰草集新美肌梦幻曲面贴膜3片 保湿补水,1.0,1.0,0.0,佰草集,1.0
1494,2016-11-10,A538981087285,【双II预售】资生堂 新透白色控霜 30ml,390.0,1.0,0.0,资生堂,390.0
24148,2016-11-09,A540190519057,【娇兰盛典】腮红亲密容和肌肤 裸妆感 自然持久玫瑰闰色腮红,420.0,1.0,0.0,娇兰,420.0
24147,2016-11-09,A540189922026,【娇兰盛典】丝柔蜜粉饼 营造细致透明妆感 柔滑细腻贴肤美颜,480.0,1.0,1.0,娇兰,480.0
16974,2016-11-05,A541166044768,L'OREAL欧莱雅卓韵霜时尚魅棕系列染发霜 富含炫闪因子蜜茶棕红棕,79.0,1.0,0.0,欧莱雅,79.0


### 2. Data Visualization

#### 2.1 Daily Sales Trend

In [22]:
result = df2.groupby('update_time').agg({'sale_count':'sum'}).to_dict()['sale_count']
c = (
    Line()
    .add_xaxis(list(result.keys()))
    .add_yaxis("销售量", list(result.values()))
    .set_series_opts(
        areastyle_opts=opts.AreaStyleOpts(opacity=0.5),
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_="max", name="最大值"),
                opts.MarkPointItem(type_="min", name="最小值"),
                opts.MarkPointItem(type_="average", name="平均值"),
            ]
        ),
    )
    .set_global_opts(title_opts=opts.TitleOpts(title="每日整体销售量走势"))
)
c.render_notebook()

#### 2.2 Which cosmetics brand sells the best?

In [23]:
dates = list(df2['update_time'].unique())
dates.reverse()
dates

[Timestamp('2016-11-05 00:00:00'),
 Timestamp('2016-11-06 00:00:00'),
 Timestamp('2016-11-07 00:00:00'),
 Timestamp('2016-11-08 00:00:00'),
 Timestamp('2016-11-09 00:00:00'),
 Timestamp('2016-11-10 00:00:00'),
 Timestamp('2016-11-11 00:00:00'),
 Timestamp('2016-11-12 00:00:00'),
 Timestamp('2016-11-13 00:00:00'),
 Timestamp('2016-11-14 00:00:00')]

In [24]:
from pyecharts import options as opts
from pyecharts.charts import Map, Timeline, Bar, Line, Pie
from pyecharts.components import Table
from pyecharts.options import ComponentTitleOpts

tl = Timeline()
tl.add_schema(
#         is_auto_play=True,
        is_loop_play=False,
        play_interval=500,
    )
for date in dates:
    item = df2[df2['update_time'] <= date].groupby('店名').agg({'sale_count': 'sum', 'sale_revenue': 'sum'}).sort_values(by='sale_count', ascending=False)[:10].sort_values(by='sale_count').to_dict()
    bar = (
        Bar()
        .add_xaxis([*item['sale_count'].keys()])
        .add_yaxis("销售量", [round(val/10000,2) for val in item['sale_count'].values()], label_opts=opts.LabelOpts(position="right", formatter='{@[1]/} 万'))
        .add_yaxis("销售额", [round(val/10000/10000,2) for val in item['sale_revenue'].values()], label_opts=opts.LabelOpts(position="right", formatter='{@[1]/} 亿元'))
        .reversal_axis()
        .set_global_opts(
            title_opts=opts.TitleOpts("累计销售量排行 TOP10")
        )
    )
    tl.add(bar, date)
tl.render_notebook()